# Complete Workflow: Library Access Equity Study

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/06-complete-workflow.ipynb)

## Learning Objectives

By the end of this notebook, you will be able to:

- Execute a complete accessibility analysis from start to finish
- Use `analyze_multiple_pois()` for multi-location comparison
- Generate professional reports with `generate_report()`
- Create publication-ready visualizations with overlays and statistics
- Export results in multiple formats

## Prerequisites

- Completed notebooks 01-05

## Research Question

**How equitable is library access in a city? Who has walkable access to public libraries?**

## Setup

In [ ]:
# Install SocialMapper (pin versions for Colab compatibility)
!pip install -q "socialmapper[routing]" "pandas<3.0" "numpy<2.0"

# IMPORTANT: After install, go to Runtime > Restart session, then skip this cell

In [ ]:
import os
import json
os.environ["SOCIALMAPPER_DEMO_MODE"] = "true"

import socialmapper
print(f"SocialMapper v{socialmapper.__version__}")

from socialmapper import (
    create_isochrone,
    get_poi,
    get_census_blocks,
    get_census_data,
    create_map,
    analyze_multiple_pois,
    generate_report
)
from IPython.display import Image, display, HTML

print("Ready for analysis!")

## Part 1: Single Location Analysis

Let's start with a comprehensive analysis of one location.

### Step 1: Define the Study Area

In [ ]:
# Study location
location = "Portland, OR"

# Create a 20-minute walking isochrone
walk_isochrone = create_isochrone(
    location=location,
    travel_time=20,
    travel_mode="walk"
)

print(f"Study Area: {location}")
print(f"Travel Mode: Walking")
print(f"Travel Time: 20 minutes")
print(f"Area Coverage: {walk_isochrone['properties']['area_sq_km']:.2f} km²")
print(f"Backend: {walk_isochrone['properties']['backend']}")

### Step 2: Find Libraries

In [ ]:
# Query for public libraries
libraries = get_poi(
    location=location,
    categories=["education"],
    travel_time=20,
    limit=50
)

print(f"Libraries found: {len(libraries)}")
print("\nLibrary Details:")
for lib in libraries[:5]:
    print(f"  - {lib['name']}: {lib['distance_km']:.2f} km from center")

### Step 3: Get Census Geography

In [ ]:
# Get census blocks within the walkable area
blocks = get_census_blocks(polygon=walk_isochrone)

print(f"Census block groups in study area: {len(blocks)}")

# Show summary
total_area = sum(b['area_sq_km'] for b in blocks)
print(f"Total census area: {total_area:.2f} km²")

### Step 4: Retrieve Demographics

In [ ]:
# Get demographic data for all blocks
geoids = [b['geoid'] for b in blocks]

census_result = get_census_data(
    location=geoids,
    variables=["population", "median_income", "median_age"]
)

print(f"Census data retrieved:")
print(f"  Year: {census_result.query_info['year']}")
print(f"  Block groups: {len(census_result.data)}")

# Combine census data with blocks (robust pattern)
for block in blocks:
    data = census_result.data.get(block['geoid'], {})
    pop = data.get('population')
    block['population'] = pop if pop is not None and pop >= 0 else 0
    
    income = data.get('median_income')
    block['median_income'] = income if income is not None and income > 0 else None
    
    age = data.get('median_age')
    block['median_age'] = age if age is not None and age > 0 else None

### Step 5: Analyze the Data

In [ ]:
# Calculate statistics with data quality tracking
total_population = sum(b['population'] for b in blocks)
incomes = [b['median_income'] for b in blocks if b['median_income']]
ages = [b['median_age'] for b in blocks if b['median_age']]

print("=" * 60)
print("STUDY RESULTS: Library Access Equity Analysis")
print("=" * 60)

print(f"\nGeographic Coverage:")
print(f"  Walkable area: {walk_isochrone['properties']['area_sq_km']:.2f} km²")
print(f"  Census block groups: {len(blocks)}")
print(f"  Libraries accessible: {len(libraries)}")

print(f"\nPopulation with Library Access:")
print(f"  Total population: {total_population:,}")
if len(blocks) > 0:
    print(f"  Average per block group: {total_population // len(blocks):,}")

if incomes:
    print(f"\nEconomic Profile:")
    print(f"  Median income range: ${min(incomes):,} - ${max(incomes):,}")
    print(f"  Average median income: ${sum(incomes)//len(incomes):,}")
    print(f"  Data available for: {len(incomes)}/{len(blocks)} blocks")

if ages:
    print(f"\nAge Profile:")
    print(f"  Median age range: {min(ages):.1f} - {max(ages):.1f} years")
    print(f"  Average median age: {sum(ages)/len(ages):.1f} years")

### Step 6: Create Publication-Ready Visualization

In [ ]:
# Prepare data for visualization
blocks_with_pop = [b for b in blocks if b['population'] > 0]

# Format library points for overlay
library_points = [
    {'lat': lib['lat'], 'lon': lib['lon'], 'name': lib['name']}
    for lib in libraries
]

# Create custom statistics
avg_income = sum(incomes) // len(incomes) if incomes else 0

stats_dict = {
    "Study Area": location,
    "Travel Time": "20-min walk",
    "Population": f"{total_population:,}",
    "Libraries": len(libraries),
    "Avg Income": f"${avg_income:,}" if avg_income else "N/A",
    "Block Groups": len(blocks)
}

# Create the map with all features
pop_map = create_map(
    data=blocks_with_pop,
    column="population",
    title="Library Access Equity Analysis - Portland, OR",
    basemap="CartoDB.Positron",
    cmap="YlGnBu",
    overlay_boundary=walk_isochrone,
    overlay_points=library_points,
    stats_dict=stats_dict,
    save_path="library_access_population.png"
)

print(f"Publication-ready map saved: {pop_map.file_path}")
display(Image(pop_map.image_data))

## Part 2: Multi-Location Comparison with analyze_multiple_pois()

Use `analyze_multiple_pois()` to compare accessibility across multiple locations efficiently.

In [ ]:
# Compare multiple locations
locations = [
    "Portland, OR",
    "Durham, NC",
    "Chapel Hill, NC"
]

# Analyze all locations at once
multi_result = analyze_multiple_pois(
    locations=locations,
    travel_time=15,
    travel_mode="walk",
    variables=["population", "median_income"],
    compare=True
)

print("Multi-Location Analysis Results:")
print("=" * 60)

In [ ]:
# Display individual location results
for loc_data in multi_result['locations']:
    print(f"\n{loc_data['location']}:")
    print(f"  Isochrone area: {loc_data['isochrone']['properties']['area_sq_km']:.2f} km²")
    print(f"  POIs found: {loc_data['poi_count']}")
    print(f"  Total population: {loc_data['demographics'].get('population', 'N/A'):,}")
    income = loc_data['demographics'].get('median_income')
    print(f"  Median income: ${income:,.0f}" if income else "  Median income: N/A")

In [ ]:
# Show comparison summary
if 'comparison' in multi_result:
    comparison = multi_result['comparison']
    print("\nComparison Summary:")
    print("-" * 60)
    
    print(f"Highest population: {comparison.get('highest_population', 'N/A')}")
    print(f"Lowest population: {comparison.get('lowest_population', 'N/A')}")
    print(f"Most POIs: {comparison.get('most_pois', 'N/A')}")
    print(f"Largest area: {comparison.get('largest_area', 'N/A')}")

## Part 3: Generate Professional Report

Use `generate_report()` to create a formatted HTML or PDF report.

In [ ]:
# Prepare analysis data for report
analysis_data = {
    "title": "Library Access Equity Analysis",
    "locations": multi_result['locations'],
    "comparison": multi_result.get('comparison', {}),
    "metadata": multi_result.get('metadata', {})
}

# Generate HTML report
html_report = generate_report(
    analysis_data=analysis_data,
    format="html",
    template="default",
    include_maps=True
)

# Save report
with open("library_access_report.html", "w") as f:
    f.write(html_report)

print("HTML report generated: library_access_report.html")
print(f"Report size: {len(html_report):,} characters")

## Part 4: Export Results in Multiple Formats

In [ ]:
# Export as GeoJSON for web mapping
geojson_result = create_map(
    data=blocks_with_pop,
    column="population",
    export_format="geojson"
)

with open("library_access.geojson", "w") as f:
    json.dump(geojson_result.geojson_data, f, indent=2)

print(f"GeoJSON exported: library_access.geojson")
print(f"Features: {len(geojson_result.geojson_data['features'])}")

In [ ]:
# Export as PDF for print
pdf_map = create_map(
    data=blocks_with_pop,
    column="population",
    title="Library Access - Portland, OR",
    overlay_boundary=walk_isochrone,
    overlay_points=library_points,
    stats_dict=stats_dict,
    export_format="pdf"
)

with open("library_access.pdf", "wb") as f:
    f.write(pdf_map.image_data)

print(f"PDF exported: library_access.pdf")
print(f"Size: {len(pdf_map.image_data):,} bytes")

In [ ]:
# Create comprehensive JSON report
report_data = {
    "study": "Library Access Equity Analysis",
    "date": "2025-01-24",
    "primary_location": {
        "name": location,
        "travel_mode": "walk",
        "travel_time_minutes": 20
    },
    "coverage": {
        "area_sq_km": walk_isochrone['properties']['area_sq_km'],
        "block_groups": len(blocks),
        "libraries": len(libraries)
    },
    "demographics": {
        "total_population": total_population,
        "avg_median_income": avg_income if avg_income else None,
        "avg_median_age": round(sum(ages)/len(ages), 1) if ages else None
    },
    "libraries": [
        {"name": lib['name'], "distance_km": round(lib['distance_km'], 2)}
        for lib in libraries[:10]  # Top 10
    ],
    "multi_location_comparison": multi_result.get('comparison', {})
}

with open("library_access_data.json", "w") as f:
    json.dump(report_data, f, indent=2)

print("Data export saved: library_access_data.json")

## Part 5: Reusable Analysis Function

In [ ]:
def analyze_amenity_access(
    location: str,
    amenity_category: str,
    travel_time: int = 20,
    travel_mode: str = "walk"
):
    """
    Complete amenity accessibility analysis for any location and category.
    
    Parameters
    ----------
    location : str
        City name (e.g., "Portland, OR")
    amenity_category : str
        POI category (e.g., "education", "healthcare", "shopping")
    travel_time : int
        Travel time in minutes
    travel_mode : str
        "walk", "bike", or "drive"
    
    Returns
    -------
    dict
        Complete analysis results
    """
    print(f"\n{'='*60}")
    print(f"{amenity_category.title()} Access Analysis: {location}")
    print(f"{'='*60}")
    
    # Step 1: Create isochrone
    print("\n[1/5] Creating isochrone...")
    isochrone = create_isochrone(
        location=location,
        travel_time=travel_time,
        travel_mode=travel_mode
    )
    print(f"      Area: {isochrone['properties']['area_sq_km']:.2f} km²")
    
    # Step 2: Find POIs
    print(f"\n[2/5] Finding {amenity_category} POIs...")
    pois = get_poi(
        location=location,
        categories=[amenity_category],
        travel_time=travel_time,
        limit=50
    )
    print(f"      Found: {len(pois)}")
    
    # Step 3: Get census blocks
    print("\n[3/5] Getting census blocks...")
    blocks = get_census_blocks(polygon=isochrone)
    print(f"      Block groups: {len(blocks)}")
    
    # Step 4: Get demographics
    print("\n[4/5] Retrieving demographics...")
    geoids = [b['geoid'] for b in blocks]
    census_result = get_census_data(
        location=geoids,
        variables=["population", "median_income"]
    )
    
    # Process data
    for block in blocks:
        data = census_result.data.get(block['geoid'], {})
        pop = data.get('population')
        block['population'] = pop if pop is not None and pop >= 0 else 0
        income = data.get('median_income')
        block['median_income'] = income if income is not None and income > 0 else None
    
    total_pop = sum(b['population'] for b in blocks)
    incomes = [b['median_income'] for b in blocks if b['median_income']]
    avg_income = sum(incomes) // len(incomes) if incomes else None
    
    # Step 5: Create visualization
    print("\n[5/5] Creating visualization...")
    blocks_valid = [b for b in blocks if b['population'] > 0]
    poi_points = [{'lat': p['lat'], 'lon': p['lon'], 'name': p['name']} for p in pois]
    
    filename = f"{location.replace(', ', '_').replace(' ', '_').lower()}_{amenity_category}.png"
    
    stats = {
        "Location": location,
        "Travel": f"{travel_time}-min {travel_mode}",
        "Population": f"{total_pop:,}",
        amenity_category.title(): len(pois),
        "Avg Income": f"${avg_income:,}" if avg_income else "N/A"
    }
    
    map_result = create_map(
        data=blocks_valid,
        column="population",
        title=f"{amenity_category.title()} Access - {location}",
        overlay_boundary=isochrone,
        overlay_points=poi_points,
        stats_dict=stats,
        save_path=filename
    )
    print(f"      Saved: {filename}")
    
    # Summary
    print(f"\n{'='*60}")
    print("RESULTS SUMMARY")
    print(f"{'='*60}")
    print(f"Location: {location}")
    print(f"Travel: {travel_time} minutes {travel_mode}ing")
    print(f"Area: {isochrone['properties']['area_sq_km']:.2f} km²")
    print(f"{amenity_category.title()}: {len(pois)}")
    print(f"Population with access: {total_pop:,}")
    if avg_income:
        print(f"Average median income: ${avg_income:,}")
    
    return {
        "location": location,
        "amenity_category": amenity_category,
        "travel_time": travel_time,
        "travel_mode": travel_mode,
        "isochrone": isochrone,
        "pois": pois,
        "blocks": blocks,
        "total_population": total_pop,
        "avg_income": avg_income,
        "map_file": filename
    }

In [ ]:
# Example: Analyze healthcare access
healthcare_result = analyze_amenity_access(
    location="Portland, OR",
    amenity_category="healthcare",
    travel_time=15
)

# Display the map
display(Image(filename=healthcare_result['map_file']))

In [ ]:
# Compare different amenity types for same location
amenities = ["education", "healthcare", "shopping"]

results = {}
for amenity in amenities:
    result = analyze_amenity_access(
        location="Portland, OR",
        amenity_category=amenity,
        travel_time=15
    )
    results[amenity] = result

# Summary comparison
print("\n" + "=" * 60)
print("AMENITY COMPARISON - Portland, OR (15-min walk)")
print("=" * 60)

for amenity, result in results.items():
    print(f"\n{amenity.title()}:")
    print(f"  POIs found: {len(result['pois'])}")
    print(f"  Population served: {result['total_population']:,}")

## Troubleshooting

### Common Issues

| Issue | Solution |
|-------|----------|
| Empty POI results | Check category name, try broader area |
| Census data missing | Use robust aggregation with None handling |
| Map not displaying | Use `Image(result.image_data)` for Colab |
| Report generation fails | Check analysis_data structure |

## Next Steps

- **[Food Desert Case Study](07-food-desert-case-study.ipynb)** - Apply these techniques to food access analysis
- Try analyzing different amenities (hospitals, parks, transit)
- Compare urban vs. suburban accessibility
- Create time-series analysis by year